# 4.8 · 非线性回归 / Nonlinear Regression

> **课程定位 / Where this fits**
> 第 8 课，**Part 4 · 监督学习：回归**。
> Lesson 8, **Part 4 · Supervised Regression**.
>
> 前面的模型（含 GLM）都是"对参数线性"的。但有时目标和特征是**真正的非线性参数关系**——指数衰减、S 形增长、正弦周期等，且**参数本身有物理/业务含义**（衰减速率、饱和上限）。这时用**非线性最小二乘**直接拟合这条曲线的参数。
> All earlier models (incl. GLM) are "linear in parameters". But sometimes the relationship is **genuinely nonlinear in parameters** — exponential decay, S-shaped growth, sinusoids — and **the parameters themselves carry physical/business meaning** (decay rate, saturation ceiling). Then use **nonlinear least squares** to fit those parameters directly.
>
> 💼 **实战/面试视角**：增长曲线预测（产品采用、疫情）、剂量反应、物理建模等场景常用；初值敏感是关键坑。
> 💼 **Practical/interview angle:** growth-curve forecasting (adoption, epidemics), dose-response, physical modeling; initial-guess sensitivity is the key gotcha.

> 💡 **面试相关 / Interview-relevant**
> - "非线性回归 vs 多项式回归区别"（出镜率 ★★★，对参数非线性 vs 线性）
> - "为什么非线性回归对初值敏感 / 局部最优"（★★★★）
> - "怎么得到参数的置信区间"（★★★）
> - "逻辑增长曲线能预测什么"（★★★）

---

## 学习目标 / Learning Objectives

1. 区分"对参数线性"和"对参数非线性"。
   Distinguish "linear in parameters" from "nonlinear in parameters".
2. 用 `scipy.optimize.curve_fit` 拟合自定义曲线。
   Fit custom curves with `scipy.optimize.curve_fit`.
3. 理解**初值敏感 / 局部最优**这个核心坑。
   Understand the core gotcha: sensitivity to initial guess / local optima.
4. 从协方差矩阵得到**参数的置信区间**。
   Get **parameter confidence intervals** from the covariance matrix.
5. 用**逻辑增长曲线**做饱和量预测。
   Use the logistic growth curve to forecast saturation.

## 目录 / TOC
1. [先建直觉 + 数据](#1)
2. [curve_fit 拟合指数衰减 ⭐](#2)
3. [初值敏感 / 局部最优 ⭐](#3)
4. [参数置信区间 ⭐](#4)
5. [逻辑增长曲线：饱和预测 ⭐](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 先建直觉 + 数据 / Intuition & Data

回忆 4.3：多项式回归 $\hat y = w_0 + w_1 x + w_2 x^2$ 虽然是曲线，但它**对参数 $w$ 是线性的**（每个 $w$ 前面都是已知的 $x^k$），所以还是线性回归。
Recall 4.3: polynomial regression $\hat y = w_0 + w_1 x + w_2 x^2$ is a curve but is **linear in the parameters $w$** (each $w$ multiplies a known $x^k$), so it's still linear regression.

**非线性回归**指的是参数以非线性方式进入模型，例如指数衰减 $y = a\,e^{-bx} + c$——参数 $b$ 被卡在指数里，没有闭式解，必须**迭代优化**。代价是：(1) 对初值敏感、可能陷局部最优；(2) 需要你**事先知道函数形式**。回报是：参数有明确含义（$a$=初始幅度，$b$=衰减率，$c$=基线）。
**Nonlinear regression** means parameters enter the model nonlinearly, e.g. exponential decay $y = a\,e^{-bx} + c$ — the parameter $b$ is trapped inside the exponential, with no closed form, requiring **iterative optimization**. The costs: (1) sensitivity to the initial guess, possible local optima; (2) you must **know the functional form** in advance. The reward: parameters have clear meaning ($a$=initial amplitude, $b$=decay rate, $c$=baseline).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)
print("scipy.optimize.curve_fit 已就绪 (非线性最小二乘求解器)")


<a id="2"></a>
## 2. curve_fit 拟合指数衰减 ⭐ / Fitting Exponential Decay

`curve_fit` 的"三件套"：**你定义的函数 + 数据 (x, y) + 初值 p0**。它用 Levenberg-Marquardt 等算法迭代，使残差平方和最小，返回最优参数 `popt` 和参数协方差 `pcov`。
The `curve_fit` recipe: **your function + data (x, y) + initial guess p0**. It iterates (e.g. Levenberg-Marquardt) to minimize the sum of squared residuals, returning best parameters `popt` and their covariance `pcov`.


In [ ]:
# 自定义模型: 指数衰减 a·e^(-bx)+c, 参数有物理含义 / the model to fit
def exp_decay(x, a, b, c):
    return a * np.exp(-b * x) + c

x = np.linspace(0, 5, 60)
true_params = [5.0, 0.8, 1.0]                      # 真实 a, b, c
y = exp_decay(x, *true_params) + rng.normal(0, 0.2, len(x))   # 真值+噪声

# curve_fit(函数, x, y, p0=初值) → 返回最优参数 popt 和协方差 pcov / fit
popt, pcov = curve_fit(exp_decay, x, y, p0=[1, 1, 1])
print(f"真实参数 true: a={true_params[0]}, b={true_params[1]}, c={true_params[2]}")
print(f"拟合参数 fit:  a={popt[0]:.3f}, b={popt[1]:.3f}, c={popt[2]:.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x, y, alpha=0.5, s=15, label="数据 data")
ax.plot(x, exp_decay(x, *popt), "r-", lw=2, label=f"拟合 {popt[0]:.1f}·e^(-{popt[1]:.1f}x)+{popt[2]:.1f}")
ax.legend(); ax.set_title("非线性回归: curve_fit 拟合指数衰减")
plt.tight_layout(); plt.show()
print("三件套: 自定义函数 + 数据 + 初值 p0 → 最优参数 popt 和协方差 pcov")


<a id="3"></a>
## 3. 初值敏感 / 局部最优 ⭐ / Sensitivity & Local Optima

这是非线性回归**最核心的坑**：因为损失函数**非凸**（不像线性回归那样只有一个最低点），迭代会停在离初值最近的那个"谷底"。如果初值离真实参数太远，就会**陷入错误的局部最优**。正弦频率拟合是出名的例子——频率初值差一点，就掉进错误的"波峰盆地"。
This is nonlinear regression's **central gotcha**: because the loss is **non-convex** (unlike linear regression's single minimum), iteration settles in whatever "valley" is nearest the start. A bad initial guess **gets stuck in a wrong local optimum**. Fitting a sinusoid's frequency is the classic example — a slightly-off frequency guess falls into the wrong "ridge basin".


In [ ]:
def sine_model(x, a, freq, phase):
    return a * np.sin(freq * x + phase)

x2 = np.linspace(0, 10, 100)
y2 = sine_model(x2, 2.0, 1.5, 0.5) + rng.normal(0, 0.2, 100)   # 真实频率 1.5

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# 好初值: 频率猜 1.4, 接近真实 1.5 → 收敛 / good guess near true frequency
popt_good, _ = curve_fit(sine_model, x2, y2, p0=[1, 1.4, 0])
axes[0].scatter(x2, y2, alpha=0.3, s=10); axes[0].plot(x2, sine_model(x2, *popt_good), "g-", lw=2, label=f"freq={popt_good[1]:.2f}")
axes[0].legend(); axes[0].set_title("好初值(freq≈1.4): 收敛到真实 1.5 ✓")
# 坏初值: 频率猜 5.0, 离真实太远 → 陷错误局部最优 / bad guess, wrong basin
popt_bad, _ = curve_fit(sine_model, x2, y2, p0=[1, 5.0, 0], maxfev=5000)
axes[1].scatter(x2, y2, alpha=0.3, s=10); axes[1].plot(x2, sine_model(x2, *popt_bad), "r-", lw=2, label=f"freq={popt_bad[1]:.2f}")
axes[1].legend(); axes[1].set_title("坏初值(freq=5): 陷局部最优, 拟合失败 ✗")
plt.tight_layout(); plt.show()
print("正弦频率拟合是出名的多局部最优问题 — 初值离真实频率太远就陷错误盆地")
print("实战对策: 用领域知识/网格搜索定初值; 或多个初值取最优(避局部陷阱)")


<a id="4"></a>
## 4. 参数置信区间 ⭐ / Parameter Confidence Intervals

非线性回归不只给点估计——`curve_fit` 返回的 `pcov`（参数协方差矩阵，来自 2.9 的 Fisher 信息/Hessian）能给出**每个参数的标准误和置信区间**。对角线开平方就是标准误，$\pm1.96\,SE$ 即 95% 置信区间（2.5）。这让你能量化"衰减率到底有多确定"。
Nonlinear regression gives more than point estimates — the `pcov` returned by `curve_fit` (the parameter covariance from Fisher information/Hessian, 2.9) yields **each parameter's standard error and CI**. Square-root the diagonal for SEs, $\pm1.96\,SE$ for a 95% CI (2.5). This quantifies "how certain is the decay rate".


In [ ]:
popt, pcov = curve_fit(exp_decay, x, y, p0=[1,1,1])
perr = np.sqrt(np.diag(pcov))         # 参数标准误 = 协方差矩阵对角线开根 / SEs from covariance diagonal

print(f"{'参数':<6} {'估计 est':>9} {'标准误 SE':>9} {'95% CI':>22} {'真值':>6}")
for name, est, se, true in zip("abc", popt, perr, true_params):
    lo, hi = est - 1.96*se, est + 1.96*se     # 95% CI = est ± 1.96·SE
    inside = "✓" if lo <= true <= hi else "✗"
    print(f"{name:<6} {est:>9.3f} {se:>9.3f} [{lo:.3f}, {hi:.3f}]  {true:>6} {inside}")
print("\npcov 来自 Fisher/Hessian(2.9); 对角线开根=SE; ±1.96·SE=95% CI(2.5)")
print("→ 非线性回归同样能做完整统计推断, 不只点估计")


<a id="5"></a>
## 5. 逻辑增长曲线：饱和预测 ⭐ / Logistic Growth: Forecasting Saturation

非线性回归的杀手应用：**S 形增长曲线**。$y = \frac{L}{1+e^{-k(x-x_0)}}$ 描述"先慢→加速→饱和"的增长（产品累计采用量、疫情累计感染、用户增长）。最有价值的是参数 **$L$（饱和上限）**——它能让你**从早期数据预测最终天花板**。
A killer application: the **S-shaped growth curve**. $y = \frac{L}{1+e^{-k(x-x_0)}}$ describes "slow → accelerate → saturate" growth (cumulative product adoption, epidemic cases, user growth). The most valuable parameter is **$L$ (the saturation ceiling)** — it lets you **forecast the final plateau from early data**.


In [ ]:
def logistic_growth(x, L, k, x0):
    return L / (1 + np.exp(-k * (x - x0)))    # L=上限, k=陡度, x0=拐点位置

x3 = np.linspace(0, 20, 80)
y3 = logistic_growth(x3, L=1000, k=0.6, x0=10) + rng.normal(0, 25, 80)   # 真实上限 1000

popt3, pcov3 = curve_fit(logistic_growth, x3, y3, p0=[800, 0.5, 8])      # 初值大致即可
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x3, y3, alpha=0.4, s=15, label="累计采用量 cumulative adoption")
ax.plot(x3, logistic_growth(x3, *popt3), "r-", lw=2,
        label=f"拟合: L={popt3[0]:.0f}, k={popt3[1]:.2f}, x0={popt3[2]:.1f}")
ax.axhline(popt3[0], color="g", ls="--", alpha=0.5, label=f"饱和值 ceiling L≈{popt3[0]:.0f}")
ax.legend(); ax.set_title("逻辑增长曲线: 预测最终饱和量(产品/疫情建模)")
plt.tight_layout(); plt.show()
print(f"模型外推: 最终饱和采用量 L = {popt3[0]:.0f} (真实 1000)")
print("逻辑增长是非线性回归的杀手应用: 从早期数据预测最终天花板(新冠/新产品建模常用)")


<a id="6"></a>
## 6. 小结 / Summary

```
非线性回归: 参数以非线性方式进入模型(如 a·e^(-bx)+c, b 在指数里); 无闭式解, 迭代优化
vs 多项式回归: 多项式对参数线性(仍是线性回归), 非线性回归对参数非线性
curve_fit 三件套: 自定义函数 + 数据 + 初值 p0; 返回 popt(参数) + pcov(协方差)
核心坑: 损失非凸 → 初值敏感, 可能陷局部最优(正弦频率经典); 多初值/领域知识/网格定初值
参数推断: SE=√diag(pcov), 95% CI=est±1.96·SE; 参数有物理/业务含义
逻辑增长曲线: S 形, L=饱和上限 → 从早期数据预测最终天花板(产品/疫情)
```

### 💡 面试速查 / Interview cheat-sheet
1. **非线性回归=对参数非线性**(多项式回归对参数线性, 仍是线性)。
   Nonlinear = nonlinear in parameters (polynomial is linear in parameters, still linear).
2. **损失非凸 → 初值敏感**, 可能陷局部最优(正弦频率经典坑)。
   Non-convex loss → initial-guess sensitive, can hit local optima.
3. **参数有明确含义**(衰减率/饱和上限), 这是用它的主要理由。
   Parameters carry meaning (decay rate / ceiling) — the main reason to use it.
4. **pcov 给参数 CI**, 同样能做统计推断。
   pcov gives parameter CIs; full inference is possible.
5. **逻辑增长**从早期数据预测最终饱和量。
   Logistic growth forecasts the final saturation from early data.

### 下一节 / Next
**4.9 支持向量回归(SVR)**——把 SVM 的"间隔"思想用到回归: ε-不敏感管道内不计误差, 加核技巧拟合非线性。
**4.9 SVR** — bring SVM's margin idea to regression: no penalty inside an ε-insensitive tube, plus the kernel trick for nonlinearity.
